# Arrow handoff fast path

This notebook demonstrates the high-performance handoff path: Arrow tables are normalized, converted to package-owned columnar batches, and passed to the batch capital calculators without materializing accepted rows as dataclasses.


In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd()
PACKAGE_ROOT = None
REPO_ROOT = None

for candidate in (HERE, *HERE.parents):
    if (candidate / "src" / "frtb_cva").exists():
        PACKAGE_ROOT = candidate
        REPO_ROOT = candidate.parent.parent if candidate.parent.name == "packages" else candidate
        break
    package_candidate = candidate / "packages" / "frtb-cva"
    if (package_candidate / "src" / "frtb_cva").exists():
        REPO_ROOT = candidate
        PACKAGE_ROOT = package_candidate
        break

if PACKAGE_ROOT is None or REPO_ROOT is None:
    raise RuntimeError("Could not locate the frtb-cva package root")

workspace_src_paths = tuple(sorted((REPO_ROOT / "packages").glob("*/src")))
for path in (
    PACKAGE_ROOT / "examples",
    PACKAGE_ROOT,
    PACKAGE_ROOT / "src",
    *workspace_src_paths,
    REPO_ROOT,
):
    text = str(path)
    if text not in sys.path:
        sys.path.insert(0, text)

try:
    from IPython.display import Markdown, display
except ImportError:

    class Markdown(str):
        pass

    def display(value: object) -> None:
        print(value)


PACKAGE_ROOT

In [ ]:
import numpy as np

from cva_notebook_data import (
    build_ba_arrow_batches,
    build_sa_arrow_batches,
    markdown_table,
    netting_set_arrow_table,
    notebook_context,
    sample_counterparties,
    sample_direct_hedge,
    sample_netting_sets,
    sample_sa_sensitivities,
)
from frtb_cva import (
    CvaMethod,
    build_cva_netting_set_batch_from_handoff,
    calculate_cva_capital,
    calculate_cva_capital_from_batches,
    normalize_cva_netting_set_arrow_table,
    validate_cva_result_reconciliation,
)

counterparties = sample_counterparties()
netting_sets = sample_netting_sets(counterparties)
hedges = (sample_direct_hedge(),)
row_context = notebook_context(method=CvaMethod.BA_CVA_FULL, run_id="cva-arrow-row-reference")
batch_context = notebook_context(method=CvaMethod.BA_CVA_FULL, run_id="cva-arrow-batch")

row_result = calculate_cva_capital(row_context, counterparties, netting_sets, hedges=hedges)
ba_pack = build_ba_arrow_batches(counterparties, netting_sets, hedges)
batch_calculation = calculate_cva_capital_from_batches(
    batch_context,
    ba_pack.counterparty_batch,
    ba_pack.netting_set_batch,
    hedges=ba_pack.hedge_batch,
)
validate_cva_result_reconciliation(batch_calculation.result)

display(
    Markdown(
        markdown_table(
            ("path", "total CVA capital", "input hash prefix"),
            (
                (
                    "row dataclass reference",
                    f"{row_result.total_cva_capital:,.2f}",
                    row_result.input_hash[:16],
                ),
                (
                    "Arrow batch",
                    f"{batch_calculation.result.total_cva_capital:,.2f}",
                    batch_calculation.result.input_hash[:16],
                ),
            ),
        )
    )
)

In [ ]:
display(
    Markdown(
        markdown_table(
            ("accepted row materialization counter", "value"),
            (
                (
                    "counterparties",
                    batch_calculation.accepted_counterparty_dataclasses_materialized,
                ),
                ("netting sets", batch_calculation.accepted_netting_set_dataclasses_materialized),
                ("hedges", batch_calculation.accepted_hedge_dataclasses_materialized),
            ),
        )
    )
)
assert batch_calculation.result.total_cva_capital == row_result.total_cva_capital

In [ ]:
table = netting_set_arrow_table(netting_sets)
handoff = normalize_cva_netting_set_arrow_table(table)
netting_set_batch = build_cva_netting_set_batch_from_handoff(handoff)
arrow_maturity_buffer = table.column("effective_maturity").chunk(0).to_numpy(zero_copy_only=True)
arrow_discount_buffer = table.column("discount_factor").chunk(0).to_numpy(zero_copy_only=True)

display(
    Markdown(
        markdown_table(
            ("column", "shares Arrow buffer", "batch writable"),
            (
                (
                    "effective_maturity",
                    np.shares_memory(netting_set_batch.effective_maturities, arrow_maturity_buffer),
                    netting_set_batch.effective_maturities.flags.writeable,
                ),
                (
                    "discount_factor",
                    np.shares_memory(netting_set_batch.discount_factors, arrow_discount_buffer),
                    netting_set_batch.discount_factors.flags.writeable,
                ),
            ),
        )
    )
)

In [ ]:
sensitivities = sample_sa_sensitivities()
sa_pack = build_sa_arrow_batches(sensitivities)
sa_batch_calculation = calculate_cva_capital_from_batches(
    notebook_context(method=CvaMethod.SA_CVA, run_id="cva-arrow-sa", sa_cva_approved=True),
    sensitivities=sa_pack.sensitivity_batch,
)
validate_cva_result_reconciliation(sa_batch_calculation.result)

display(
    Markdown(
        markdown_table(
            ("metric", "value"),
            (
                ("SA-CVA total", f"{sa_batch_calculation.result.total_cva_capital:,.2f}"),
                ("risk-class records", len(sa_batch_calculation.result.sa_cva_risk_class_capitals)),
                (
                    "sensitivity dataclasses materialized",
                    sa_batch_calculation.accepted_sensitivity_dataclasses_materialized,
                ),
                (
                    "hedge dataclasses materialized",
                    sa_batch_calculation.accepted_hedge_dataclasses_materialized,
                ),
            ),
        )
    )
)